# Experiment 2 (Universal Model, Leave-One-Dyad-Out) — Colab runner

This notebook runs Experiment 2 exactly as it would run locally. It ships and executes the real
`src/models.py` and `src/experiments/exp2_universal.py` from the project repository, unmodified —
this notebook contains no modeling logic of its own. It is a thin wrapper: unpack, pin dependencies,
redirect the checkpoint directory to Google Drive, optionally widen `GridSearchCV` parallelism, call
`exp2_universal.run()`, download the outputs.

**What you need to do:**

1. Upload `exp2_colab_package.zip` to the **root** of your Google Drive (`My Drive`, not a subfolder).
2. Open this notebook in Google Colab.
3. **Runtime → Change runtime type → CPU.** Do **not** select a GPU runtime. XGBoost's GPU histogram
   builder produces different split decisions than the CPU builder — a different algorithm, not just a
   faster one. Experiment 1 (the baseline this experiment is compared against) ran entirely on CPU, so
   running Experiment 2 on a GPU would introduce a confound into the very comparison the experiment
   exists to make.
4. Run every cell top to bottom, in order.
5. **The run takes hours.** Leave the browser tab open — Colab disconnects after ~90 minutes of
   *browser* idleness (not compute idleness), so keep the tab visible/active.
6. **If you get disconnected, just run every cell again from the top.** The checkpoint directory lives
   on Google Drive, not on the disposable Colab disk, so a re-run picks up any completed folds and
   permutations exactly where it left off rather than starting over.
7. The last cell downloads two files. Put them here on your local machine:
   ```
   C:\Users\Alber\CN4\reveriehacks26\results\exp2_universal.json
   C:\Users\Alber\CN4\reveriehacks26\results\exp2_universal.md
   ```


In [ ]:
!pip install -q scikit-learn==1.8.0 xgboost==3.4.1 numpy==2.3.2 scipy==1.17.1 pandas==2.3.2 pyarrow


**Now do Runtime → Restart session, then continue from Cell 4. Do not skip this.**

Pinning `numpy`/`pandas` to versions different from what Colab ships requires a kernel restart before
the new versions are importable — `pip install` alone does not hot-swap an already-imported package.

**Why the pins matter, and why `requirements.txt` is no help:** the repo's `requirements.txt` is
unpinned (`pandas openpyxl numpy scipy pyarrow scikit-learn xgboost torch`, no versions). The pins
above come from the *measured local environment* that produced Experiment 1's results, confirmed
independently by `exp1_baseline.json`'s own `meta.sklearn_version` field (`1.8.0`). Colab ships an
older scikit-learn by default (1.6.x at time of writing). `src/models.py` was written against
scikit-learn 1.8's `penalty`/`l1_ratio` API and uses `dual="auto"` and `StratifiedGroupKFold`. Running
Experiment 2 on a different scikit-learn than Experiment 1 would put a version confound inside the
exp1-vs-exp2 comparison itself. Pin, do not shrug.

`torch` and `openpyxl` are deliberately **not** installed — both are confirmed unused by
`exp2_universal.py` and `models.py`, and `torch` is a large, slow install that buys nothing here.

---

**Restart reminder, repeated, because it's easy to scroll past: restart the runtime now
(Runtime → Restart session) before running Cell 4.** If you already restarted, continue.


In [ ]:
import os
import sys

print("Python:", sys.version)
print("CPU count:", os.cpu_count())

try:
    import psutil
    total_ram_gb = psutil.virtual_memory().total / (1024 ** 3)
    print(f"Total RAM: {total_ram_gb:.1f} GB")
except ImportError:
    try:
        with open("/proc/meminfo") as f:
            for line in f:
                if line.startswith("MemTotal:"):
                    kb = int(line.split()[1])
                    print(f"Total RAM: {kb / (1024 ** 2):.1f} GB")
                    break
    except FileNotFoundError:
        print("Total RAM: could not determine (no psutil, no /proc/meminfo)")

import sklearn, numpy, scipy, pandas, xgboost
print("scikit-learn:", sklearn.__version__)
print("numpy:", numpy.__version__)
print("scipy:", scipy.__version__)
print("pandas:", pandas.__version__)
print("xgboost:", xgboost.__version__)

gpu_visible = False
try:
    import subprocess
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    gpu_visible = (result.returncode == 0)
except FileNotFoundError:
    gpu_visible = False
print("GPU visible (nvidia-smi):", gpu_visible)
if gpu_visible:
    print("WARNING: a GPU is visible. This experiment is designed to run on CPU only — "
          "see Cell 1, point 3. Do not enable GPU-aware XGBoost config.")

assert sklearn.__version__ == "1.8.0", (
    f"scikit-learn version mismatch: got {sklearn.__version__}, expected 1.8.0. "
    "This is not a cosmetic difference -- a different sklearn version can silently produce a "
    "differently-tuned model and invalidate the comparison against Experiment 1. "
    "Did you forget to restart the runtime after Cell 2?"
)
print("sklearn version check: OK (1.8.0)")

if os.cpu_count() is not None and os.cpu_count() <= 2:
    print("NOTE: this runtime has <=2 vCPUs. Raising GRID_N_JOBS above 2 later in this notebook "
          "will not speed anything up on this runtime -- the parallelism win will be small.")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
import zipfile
from pathlib import Path

DRIVE_ZIP = Path("/content/drive/MyDrive/exp2_colab_package.zip")
LOCAL_ZIP = Path("/content/exp2_colab_package.zip")
EXTRACT_ROOT = Path("/content")
REPO_ROOT = Path("/content/reveriehacks26")

assert DRIVE_ZIP.exists(), (
    f"{DRIVE_ZIP} not found. Upload exp2_colab_package.zip to the ROOT of your Google Drive "
    "(My Drive, not a subfolder) and re-run this cell."
)

print("Copying zip from Drive to local Colab disk (faster to unzip from local disk)...")
shutil.copy(DRIVE_ZIP, LOCAL_ZIP)

print("Unzipping...")
with zipfile.ZipFile(LOCAL_ZIP) as zf:
    zf.extractall(EXTRACT_ROOT)

# Verify the manifest: all files present, and the parquet is byte-exact (a truncated Drive
# upload would otherwise fail silently until hours into the run).
EXPECTED = {
    "src/models.py": None,
    "src/experiments/__init__.py": None,
    "src/experiments/exp2_universal.py": None,
    "data/processed/trial_table.csv": None,
    "data/processed/features/single_brain.parquet": 318202023,
    "data/processed/features/feature_dictionary.csv": None,
    "results/exp1_baseline.json": None,
    "results/exp1_baseline.md": None,
    "results/gate.json": None,
    "results/trial_count_gate.md": None,
    "results/frozen_hypotheses.md": None,
}

for rel, expected_size in EXPECTED.items():
    p = REPO_ROOT / rel
    assert p.exists(), f"Missing expected file after unzip: {p}"
    if expected_size is not None:
        actual = p.stat().st_size
        assert actual == expected_size, (
            f"{p} is {actual} bytes, expected {expected_size}. "
            "This usually means the Drive upload was truncated or interrupted -- "
            "re-upload exp2_colab_package.zip to Drive and re-run from Cell 5."
        )

print(f"All {len(EXPECTED)} manifest files present and verified.")
print("Repo root:", REPO_ROOT)


In [ ]:
import sys
import functools
from pathlib import Path

sys.path.insert(0, '/content/reveriehacks26/src')

import models as M
import experiments.exp2_universal as E2

GRID_N_JOBS = 2          # see the markdown cell below before changing this

_OrigGSCV = M.GridSearchCV
def _gscv(*args, **kwargs):
    kwargs["n_jobs"] = GRID_N_JOBS
    return _OrigGSCV(*args, **kwargs)
M.GridSearchCV = _gscv

E2.CHECKPOINT_DIR = Path('/content/drive/MyDrive/exp2_colab/checkpoints')
E2.CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("GridSearchCV n_jobs ->", GRID_N_JOBS)
print("checkpoints ->", E2.CHECKPOINT_DIR)
print("results will be written to ->", E2.RESULTS_DIR)


**Why `GridSearchCV`'s `n_jobs` is overridden here, and why it's safe:**

The local run pinned `n_jobs=1` in `src/models.py` as a workaround for a Windows-specific failure:
this harness kills detached background bash tasks on a timer, and scikit-learn's `loky` process pool
orphaned its worker processes when the parent process died, producing a multi-hour resource-exhaustion
cascade. **Colab has no analogous failure mode.** A notebook cell runs in the foreground of one
persistent kernel; the cell completes, raises, or the user interrupts it, and a Colab runtime
disconnect tears down the entire container — including every child process. Nothing is left to orphan.

**`n_jobs` changes scheduling, not arithmetic.** `GridSearchCV` aggregates candidate scores
deterministically regardless of worker count, and every estimator's `random_state` is pinned to `SEED`
(`models.py`). Results are expected to be identical to a serial run — and the driver's own
reproducibility validation (the deliberate cache-bypassed re-run inside `run_validations()`) will
catch it if they are not.

**Why `GRID_N_JOBS` defaults to 2, not `-1`:**

1. A standard Colab CPU runtime has only 2 vCPUs (see Cell 4's printed `os.cpu_count()`). `-1` and `2`
   are the same thing there. If you have a high-CPU runtime, you may raise `GRID_N_JOBS` in Cell 6 to
   `os.cpu_count()`.
2. `RandomForestClassifier` and `XGBClassifier` already set `n_jobs=-1` *inside* each fit. Setting
   `GridSearchCV(n_jobs=-1)` on top of that oversubscribes the CPU and can be **slower** than serial —
   values above roughly 4 can slow down the random-forest and XGBoost families specifically. The
   linear families (logistic regression, SVM) are single-threaded per fit and are the ones that
   genuinely benefit from wider `GridSearchCV` parallelism.

---

**Cell 8, next, is the multi-hour cell.** Output streams as the run progresses (per-fold and
per-family progress is printed with `flush=True` throughout, by the driver itself). Colab disconnects
after ~90 minutes of *browser* tab idleness, not compute idleness — keep the tab visible. If you do
get disconnected, see Cell 1, point 6: just run every cell again from the top.


In [ ]:
E2.run()


In [ ]:
import json
from pathlib import Path

RESULTS_JSON = E2.RESULTS_DIR / "exp2_universal.json"
RESULTS_MD = E2.RESULTS_DIR / "exp2_universal.md"

assert RESULTS_JSON.exists() and RESULTS_JSON.stat().st_size > 0, f"{RESULTS_JSON} missing or empty"
assert RESULTS_MD.exists() and RESULTS_MD.stat().st_size > 0, f"{RESULTS_MD} missing or empty"

with open(RESULTS_JSON) as f:
    results = json.load(f)

print("=== validations ===")
print(json.dumps(results.get("validations", {}), indent=2))

exp2 = results.get("experiments", {}).get("exp2", results.get("exp2", {}))
print()
print("=== headline ===")
print(json.dumps(exp2, indent=2, default=str)[:3000])

per_dyad = None
for key in ("per_dyad_scores", "per_dyad", "dyad_scores"):
    if isinstance(exp2, dict) and key in exp2:
        per_dyad = exp2[key]
        break
if per_dyad is not None:
    print()
    print("Number of per-dyad scores:", len(per_dyad))
    assert len(per_dyad) == 12, f"Expected 12 per-dyad scores, found {len(per_dyad)}"
else:
    print()
    print("NOTE: could not auto-locate the per-dyad scores block under a known key name -- "
          "inspect the printed headline block above and the validations block for the full "
          "per-dyad breakdown before trusting this run.")

print()
print("Output files verified:", RESULTS_JSON, RESULTS_MD)


In [ ]:
import zipfile
from pathlib import Path
from google.colab import files

DOWNLOAD_ZIP = Path("/content/exp2_results.zip")
with zipfile.ZipFile(DOWNLOAD_ZIP, "w") as zf:
    zf.write(RESULTS_JSON, arcname="exp2_universal.json")
    zf.write(RESULTS_MD, arcname="exp2_universal.md")

# Backup to Drive too, so a failed browser download does not cost the run.
BACKUP_DIR = Path("/content/drive/MyDrive/exp2_colab")
BACKUP_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy(RESULTS_JSON, BACKUP_DIR / "exp2_universal.json")
shutil.copy(RESULTS_MD, BACKUP_DIR / "exp2_universal.md")
print("Backed up to:", BACKUP_DIR)

files.download(str(DOWNLOAD_ZIP))


**Where the downloaded files go, on your local machine:**

```
C:\Users\Alber\CN4\reveriehacks26\results\exp2_universal.json
C:\Users\Alber\CN4\reveriehacks26\results\exp2_universal.md
```

Unzip `exp2_results.zip` (downloaded by the previous cell) and place both files at the paths above,
overwriting nothing else in `results/`.

**One expected, benign difference:** the `meta` block inside `exp2_universal.json` will report a
Linux platform and Colab's library versions, not Windows. That is correct — it reflects where the
experiment actually ran. It is not a provenance error.
